# RAG Notebook — IndicBARTSS (Medium, CPU-friendly, Kannada-native)

**Model:** `ai4bharat/IndicBARTSS`

- **Size:** ~245M parameters — medium, comfortable on CPU
- **Type:** Seq2Seq (encoder-decoder, mBART architecture)
- **Kannada:** Native support — Kannada script, no conversion needed
- **Speed on CPU:** ~3-8 seconds per answer
- **License:** MIT — fully open
- **No GPU needed, no login needed**

**Pipeline:** Question → FAISS + Cross-encoder reranking (top-3) → IndicBARTSS generates Kannada answer

**Metrics:** BLEU · BERTScore F1 · Answer Similarity · Context Relevance · Faithfulness

## Files needed
```
/content/rag_eval_set.json
/content/drive/MyDrive/AgriQADataset/AgriQADataset/Central Govt Schemes/
/content/drive/MyDrive/AgriQADataset/AgriQADataset/Karnataka State Schemes/
```
**Runtime: CPU is fine. GPU will be faster but not required.**

## ── CELL 1 ── Install packages

In [1]:
!pip install -q transformers==4.46.0 sentencepiece accelerate
!pip install -q faiss-cpu sentence-transformers
!pip install -q sacrebleu bert-score
print('✅ Packages installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
✅ Packages installed


## ── CELL 2 ── Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


## ── CELL 3 ── Paths and config

In [3]:
import os, json, re, glob
import numpy as np
import torch

EVAL_JSON     = '/content/rag_eval_set.json'
CENTRAL_DIR   = '/content/drive/MyDrive/AgriQADataset/AgriQADataset/Central Govt Schemes'
KARNATAKA_DIR = '/content/drive/MyDrive/AgriQADataset/AgriQADataset/Karnataka State Schemes'
RESULTS_DIR   = '/content/drive/MyDrive/AgriQA/rag_comparison_results'

RETRIEVER_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
RERANKER_MODEL  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

# IndicBARTSS — medium, CPU-friendly Kannada seq2seq model
GEN_MODEL       = 'ai4bharat/IndicBARTSS'
KANNADA_TAG     = '<2kn>'   # IndicBARTSS Kannada language tag

CHUNK_SIZE    = 400
CHUNK_OVERLAP = 80
TOP_K         = 3
RERANK_POOL   = 10

# CPU or GPU — works on both
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU — expect ~3-8 sec per answer')

os.makedirs(RESULTS_DIR, exist_ok=True)
assert os.path.exists(EVAL_JSON),     f'Not found: {EVAL_JSON}'
assert os.path.exists(CENTRAL_DIR),   f'Not found: {CENTRAL_DIR}'
assert os.path.exists(KARNATAKA_DIR), f'Not found: {KARNATAKA_DIR}'
print('✅ Paths verified')

Device : cpu
Running on CPU — expect ~3-8 sec per answer
✅ Paths verified


## ── CELL 4 ── Load and chunk scheme files

In [4]:
def read_scheme(path):
    with open(path, encoding='utf-8') as f:
        raw = f.read().replace('\r\n','\n').replace('\r','\n')
    txt = re.sub(r'"([^"]{2,40})":', r'\1:', raw)
    txt = re.sub(r'["\[\]{}]', ' ', txt)
    txt = re.sub(r',\s*\n', '\n', txt)
    txt = re.sub(r'[ \t]+', ' ', txt)
    txt = re.sub(r'\n{3,}', '\n\n', txt)
    return txt.strip()

def chunk_text(text, scheme_no):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        if end < len(text):
            nb = text.rfind('\n', start, end)
            if nb > start + CHUNK_SIZE // 2:
                end = nb
        chunk = text[start:end].strip()
        if len(chunk) > 30:
            chunks.append({'scheme_no': scheme_no,
                           'chunk_id': f'{scheme_no}_{len(chunks)}',
                           'text': chunk})
        start = end - CHUNK_OVERLAP if end < len(text) else len(text)
    return chunks

file_map = {}
for d in [CENTRAL_DIR, KARNATAKA_DIR]:
    for p in glob.glob(os.path.join(d, '*.txt')):
        b = os.path.basename(p).replace('.txt','')
        if b.isdigit():
            file_map[int(b)] = p

all_chunks, full_ctx_map = [], {}
for sno in sorted(file_map):
    txt = read_scheme(file_map[sno])
    full_ctx_map[sno] = txt
    all_chunks.extend(chunk_text(txt, sno))

print(f'✅ {len(file_map)} scheme files → {len(all_chunks)} chunks')

✅ 80 scheme files → 720 chunks


## ── CELL 5 ── FAISS index + cross-encoder reranker

In [5]:
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print('Loading retriever …')
retriever = SentenceTransformer(RETRIEVER_MODEL)
embeddings = retriever.encode(
    [c['text'] for c in all_chunks],
    batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
print(f'✅ FAISS index — {index.ntotal} vectors')

print('Loading cross-encoder reranker …')
rerank_tok   = AutoTokenizer.from_pretrained(RERANKER_MODEL)
rerank_model = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL)
rerank_model.eval().to(device)
print('✅ Reranker loaded')

def retrieve(question, top_k=TOP_K, pool=RERANK_POOL):
    """
    Step 1 — FAISS bi-encoder: fetch top 'pool' candidates
    Step 2 — Cross-encoder reranker: rescore, return top 'top_k'
    """
    q = retriever.encode([question], normalize_embeddings=True,
                         convert_to_numpy=True).astype(np.float32)
    scores, idxs = index.search(q, pool)
    candidates = []
    for sc, idx in zip(scores[0], idxs[0]):
        c = dict(all_chunks[idx])
        c['faiss_score'] = float(sc)
        candidates.append(c)
    pairs = [[question, c['text']] for c in candidates]
    enc   = rerank_tok(pairs, padding=True, truncation=True,
                       max_length=512, return_tensors='pt').to(device)
    with torch.no_grad():
        rerank_scores = rerank_model(**enc).logits.squeeze(-1).cpu().numpy()
    for c, sc in zip(candidates, rerank_scores):
        c['score'] = float(sc)
    candidates.sort(key=lambda x: x['score'], reverse=True)
    return candidates[:top_k]

# Quick test
test = retrieve('ಬೆಳೆ ವಿಮೆ ಯೋಜನೆ')
print(f'Test: scheme={test[0]["scheme_no"]} rerank_score={test[0]["score"]:.3f}')

Loading retriever …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

✅ Reranker loaded
Test: scheme=79 rerank_score=7.814


## ── CELL 6 ── Load IndicBARTSS

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f'Loading {GEN_MODEL} …')

# IndicBARTSS uses AlbertTokenizer internally
gen_tok = AutoTokenizer.from_pretrained(
    GEN_MODEL,
    do_lower_case=False,
    use_fast=False,
    keep_accents=True,
)

gen_model = AutoModelForSeq2SeqLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float32,   # float32 for CPU stability
)
gen_model.to(device)
gen_model.eval()

# Get the Kannada language token id
kannada_id = gen_tok.convert_tokens_to_ids(KANNADA_TAG)

param_count = sum(p.numel() for p in gen_model.parameters()) / 1e6
print(f'✅ IndicBARTSS loaded — {param_count:.0f}M parameters')
print(f'   Kannada token id : {kannada_id}')
print(f'   Device           : {device}')

Loading ai4bharat/IndicBARTSS …


tokenizer_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/976M [00:00<?, ?B/s]

✅ IndicBARTSS loaded — 244M parameters
   Kannada token id : 64008
   Device           : cpu


## ── CELL 7 ── Load evaluation set

In [ ]:
with open(EVAL_JSON, encoding='utf-8') as f:
    eval_set = json.load(f)
print(f'✅ {len(eval_set)} evaluation questions')
for q in eval_set:
    print(f'  {q["qid"]} | scheme={q["scheme_no"]} | {q["question"]}')

## ── CELL 8 ── Define 5 metrics

In [8]:
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn

_bleu = BLEU(effective_order=True)

def compute_all_metrics(predictions, references, questions, contexts):
    """
    5 metrics:
    1. BLEU              — n-gram overlap (sacrebleu)
    2. BERTScore F1      — semantic similarity (lang=kn)
    3. Answer Similarity — cosine(pred, gold) [RAGAS-style]
    4. Context Relevance — cosine(question, context) [RAGAS-style]
    5. Faithfulness      — cosine(pred, context) [RAGAS-style]
    """
    bleu_scores = [_bleu.sentence_score(p, [r]).score
                   for p, r in zip(predictions, references)]
    _, _, F1 = bert_score_fn(predictions, references, lang='kn', verbose=False)
    bert_scores = F1.numpy().tolist()

    pred_emb = retriever.encode(predictions, normalize_embeddings=True, convert_to_numpy=True)
    ref_emb  = retriever.encode(references,  normalize_embeddings=True, convert_to_numpy=True)
    q_emb    = retriever.encode(questions,   normalize_embeddings=True, convert_to_numpy=True)
    ctx_emb  = retriever.encode(contexts,    normalize_embeddings=True, convert_to_numpy=True)

    ans_sim  = (pred_emb * ref_emb).sum(axis=1).tolist()
    ctx_rel  = (q_emb   * ctx_emb).sum(axis=1).tolist()
    faithful = (pred_emb * ctx_emb).sum(axis=1).tolist()

    def avg(lst): return round(float(np.mean(lst)), 4)
    def rnd(lst): return [round(float(x), 4) for x in lst]

    return {
        'bleu':              avg(bleu_scores),
        'bertscore_f1':      avg(bert_scores),
        'answer_similarity': avg(ans_sim),
        'context_relevance': avg(ctx_rel),
        'faithfulness':      avg(faithful),
        'bleu_per_q':        rnd(bleu_scores),
        'bert_per_q':        rnd(bert_scores),
        'ans_sim_per_q':     rnd(ans_sim),
        'ctx_rel_per_q':     rnd(ctx_rel),
        'faithful_per_q':    rnd(faithful),
    }

print('✅ 5 metrics defined')

✅ 5 metrics defined


## ── CELL 9 ── IndicBARTSS inference function

IndicBARTSS is a seq2seq model (like mBART).
Input format: `question + context </s> <2kn>` (Kannada tag tells the model to generate in Kannada)
Output: Kannada answer generated by the decoder.

In [10]:
def indicbart_answer(question, context):
    input_text = f'ಪ್ರಶ್ನೆ: {question} \nಮಾಹಿತಿ: {context[:600]} </s> {KANNADA_TAG}'

    inputs = gen_tok(
        input_text,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=False,
    ).to(device)

    # IndicBARTSS tokenizer adds token_type_ids but the model doesn't use them
    inputs.pop('token_type_ids', None)

    with torch.no_grad():
        out_ids = gen_model.generate(
            **inputs,
            forced_bos_token_id=kannada_id,
            max_new_tokens=100,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    answer = gen_tok.decode(out_ids[0], skip_special_tokens=True).strip()
    return answer if answer else 'ಈ ಮಾಹಿತಿ ಲಭ್ಯವಿಲ್ಲ'

## ── CELL 10 ── Run RAG on all 15 questions

In [11]:
import time

predictions, references, questions_list, contexts_list = [], [], [], []
rows = []
total_time = 0

for qa in eval_set:
    hits    = retrieve(qa['question'])
    context = '\n'.join(h['text'] for h in hits)
    top_ctx = hits[0]['text']

    t0     = time.time()
    answer = indicbart_answer(qa['question'], context)
    elapsed = time.time() - t0
    total_time += elapsed

    predictions.append(answer)
    references.append(qa['answer'])
    questions_list.append(qa['question'])
    contexts_list.append(top_ctx)

    rows.append({
        'qid':               qa['qid'],
        'scheme_no':         qa['scheme_no'],
        'question':          qa['question'],
        'gold':              qa['answer'],
        'predicted':         answer,
        'top_scheme_hit':    hits[0]['scheme_no'],
        'retrieval_correct': qa['scheme_no'] == hits[0]['scheme_no'],
        'rerank_score':      round(hits[0]['score'], 4),
        'inference_sec':     round(elapsed, 2),
    })
    print(f'{qa["qid"]} ✓ ({elapsed:.1f}s)  →  {answer[:80]}')

print(f'\n✅ Done — total time: {total_time:.0f}s  |  avg per question: {total_time/len(eval_set):.1f}s')

E01 ✓ (16.3s)  →  ಕಾಮನ್ ವೆಲ್ತ್ ಗೇಮ್ಸ್ 2018: ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಉದ್ದೇಶ ಏನು? ಮಾಹಿತಿ: ವಿಧಾನ
E02 ✓ (16.5s)  →  ಕಾಮರ್ಿಕರು: PMFBY ಯೋಜನೆಗೆ ಅರ್ಜಿ ಸಲ್ಲಿಸಲು ಯಾವ ದಾಖಲೆಗಳು ಬೇಕು? ಮಾಹಿತಿ: ರೈತರು ಕೃಷಿ ಭೂ
E03 ✓ (19.0s)  →  চাঞ্চল্যಕರ ಮಾಹಿತಿ: ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಸಹಾಯವಾಣಿ ಸಂಖ್ಯೆ ಏನು? ಮಾಹಿತಿ: ಾಲೂ
E04 ✓ (17.7s)  →  हंगामा: ಪ್ರಧಾನಮಂತ್ರಿ ಕೃಷಿ ಸಿಂಚಾಯಿ ಯೋಜನೆ ಯಾವ ವರ್ಷ ಆರಂಭವಾಯಿತು? ಮಾಹಿತಿ: ವುದು ಹಾಗೂ ಸ
E05 ✓ (17.6s)  →  हंगामा: PMKSY ಯೋಜನೆಯ ಮುಖ್ಯ ಉದ್ದೇಶ ಏನು? ಮಾಹಿತಿ: ಯೋಜನೆ_ಸಂಖ್ಯೆ: 2 ಯೋಜನೆ_ಹೆಸರು: ಪ್ರಧ
E06 ✓ (16.7s)  →  रोख ಹಣ: PM-KISAN ಯೋಜನೆಯಡಿ ರೈತರಿಗೆ ಎಷ್ಟು ಹಣ ಸಿಗುತ್ತದೆ? ಮಾಹಿತಿ: ೈನ್ ಪೋರ್ಟಲ್ ( http
E07 ✓ (17.3s)  →  ಕಾಮರ್ಿಕರು: ಪ್ರಧಾನಮಂತ್ರಿ ಕಿಸಾನ್ ಸಮ್ಮಾನ್ ನಿಧಿ ಯೋಜನೆಗೆ ಯಾರು ಅರ್ಹರು? ಮಾಹಿತಿ: ರಕ್ಷಿತವ
E08 ✓ (17.8s)  →  हंगामा: ರೈತ ಸಿರಿ ಯೋಜನೆ ಯಾವ ರಾಜ್ಯದಲ್ಲಿ ಜಾರಿಯಲ್ಲಿದೆ? ಮಾಹಿತಿ: ್ಪಾದನೆ ಕುಸಿಯುತ್ತದೆ. ಇ
E09 ✓ (17.1s)  →  ಕಾಮನ್ ವೆಲ್ತ್ ಕ್ರೀಡಾಕೂಟದ ಬಗ್ಗೆ ಮಾಹಿತಿ: ರೈತ ಸಿರಿ ಯೋಜನೆಯ ಉದ್ದೇಶ ಏನು? ಮಾಹಿತಿ: ು ಉತ್ತ
E10 ✓ (16.4s)  →  ಕಾಮರ್ಿಕರಿಗೆ ಮಾಹಿತಿ: ಕರ್ನಾಟಕ ರೈತ ಸುರಕ್ಷಾ ವಿಮೆ ಯೋಜನೆ ಏನು ಒದಗಿಸುತ್ತದೆ? ಮಾಹಿತಿ: ರೈತ 
E11 ✓ (16.

## ── CELL 11 ── Compute all 5 metrics

In [12]:
print('Computing metrics …')
metrics = compute_all_metrics(predictions, references, questions_list, contexts_list)

print()
print('=' * 58)
print('  IndicBARTSS RAG — RESULTS  (CPU-friendly, 245M)')
print('=' * 58)
print(f'  BLEU                        : {metrics["bleu"]}')
print(f'  BERTScore F1                : {metrics["bertscore_f1"]}')
print(f'  Answer Similarity  (RAGAS)  : {metrics["answer_similarity"]}')
print(f'  Context Relevance  (RAGAS)  : {metrics["context_relevance"]}')
print(f'  Faithfulness       (RAGAS)  : {metrics["faithfulness"]}')
print('=' * 58)

Computing metrics …


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


  IndicBARTSS RAG — RESULTS  (CPU-friendly, 245M)
  BLEU                        : 1.6402
  BERTScore F1                : 0.6699
  Answer Similarity  (RAGAS)  : 0.6782
  Context Relevance  (RAGAS)  : 0.8104
  Faithfulness       (RAGAS)  : 0.9117


## ── CELL 12 ── Per-question breakdown

In [13]:
import pandas as pd

df = pd.DataFrame(rows)
df['bleu']     = metrics['bleu_per_q']
df['bert']     = metrics['bert_per_q']
df['ans_sim']  = metrics['ans_sim_per_q']
df['ctx_rel']  = metrics['ctx_rel_per_q']
df['faithful'] = metrics['faithful_per_q']

print('Per-question scores:')
print(df[['qid','bleu','bert','ans_sim','ctx_rel','faithful',
          'retrieval_correct','inference_sec']].to_string(index=False))
print(f'\nRetrieval accuracy : {df["retrieval_correct"].mean()*100:.1f}%')
print(f'Avg inference time : {df["inference_sec"].mean():.1f}s/question')

Per-question scores:
qid   bleu   bert  ans_sim  ctx_rel  faithful  retrieval_correct  inference_sec
E01 2.1833 0.6934   0.5826   0.7540    0.6477              False          16.27
E02 0.7408 0.6452   0.5471   0.8136    0.7468              False          16.48
E03 0.0000 0.5200   0.3197   0.9037    0.9397              False          18.97
E04 7.5304 0.7167   0.7575   0.8829    0.9691              False          17.74
E05 2.3298 0.6680   0.4415   0.6550    0.9508               True          17.65
E06 1.1442 0.7012   0.6189   0.6547    0.8803              False          16.71
E07 0.7105 0.6582   0.9067   0.8966    0.9726              False          17.33
E08 2.5664 0.7152   0.7408   0.8785    0.9388              False          17.84
E09 0.8943 0.7148   0.8309   0.8184    0.9573              False          17.09
E10 0.7740 0.6372   0.8860   0.8946    0.9844              False          16.42
E11 1.8095 0.7195   0.5263   0.6567    0.7977              False          16.68
E12 0.9148 0.6750  

## ── CELL 13 ── Sample predictions

In [14]:
print('Sample predictions (first 5):')
print('=' * 70)
for _, row in df.head(5).iterrows():
    print(f'QID  : {row["qid"]}')
    print(f'Q    : {row["question"]}')
    print(f'Gold : {row["gold"][:100]}')
    print(f'Pred : {row["predicted"][:100]}')
    print(f'BLEU={row["bleu"]:.3f}  BERT={row["bert"]:.3f}  AnsS={row["ans_sim"]:.3f}')
    print('-' * 70)

Sample predictions (first 5):
QID  : E01
Q    : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಉದ್ದೇಶ ಏನು?
Gold : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆವು ಬೆಳೆ ಹಾನಿಯಾದ ಸಂದರ್ಭಗಳಲ್ಲಿ ರೈತರಿಗೆ ವಿಮೆಯ ಮೂಲಕ ಆರ್ಥಿಕ ರಕ್ಷಣೆ ಒದಗಿಸಲು ರೂ
Pred : ಕಾಮನ್ ವೆಲ್ತ್ ಗೇಮ್ಸ್ 2018: ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಉದ್ದೇಶ ಏನು? ಮಾಹಿತಿ: ವಿಧಾನವನ್ನು ಉತ್ತೇಜಿಸಲು ಈ ಯ
BLEU=2.183  BERT=0.693  AnsS=0.583
----------------------------------------------------------------------
QID  : E02
Q    : PMFBY ಯೋಜನೆಗೆ ಅರ್ಜಿ ಸಲ್ಲಿಸಲು ಯಾವ ದಾಖಲೆಗಳು ಬೇಕು?
Gold : ಆಧಾರ್ ಕಾರ್ಡ್, ಬ್ಯಾಂಕ್ ಖಾತೆ ವಿವರ, ಭೂ ದಾಖಲೆ (ಪಹಣಿ), ಬೆಳೆ ವಿವರ, FRUITS ನೋಂದಣಿ ಮಾಹಿತಿ (ಅಗತ್ಯವಿದ್ದರೆ), ಕಂ
Pred : ಕಾಮರ್ಿಕರು: PMFBY ಯೋಜನೆಗೆ ಅರ್ಜಿ ಸಲ್ಲಿಸಲು ಯಾವ ದಾಖಲೆಗಳು ಬೇಕು? ಮಾಹಿತಿ: ರೈತರು ಕೃಷಿ ಭೂಮಿ ಹೊಂದಿರುವ ರೈತರು ಪಹ
BLEU=0.741  BERT=0.645  AnsS=0.547
----------------------------------------------------------------------
QID  : E03
Q    : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಸಹಾಯವಾಣಿ ಸಂಖ್ಯೆ ಏನು?
Gold : 14447 ಮತ್ತು 1800-180-1551
Pred : চাঞ্চল্যಕರ ಮಾಹಿತಿ: ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಸಹಾಯವಾಣಿ ಸಂಖ್ಯೆ ಏನು? ಮಾಹಿತಿ: ಾಲೂಕು ಕೃಷಿ ಇಲಾಖೆ

## ── CELL 14 ── Save results

In [15]:
csv_path = f'{RESULTS_DIR}/indicbartss_rag_results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ CSV saved → {csv_path}')

result_dict = {
    'model':             GEN_MODEL,
    'type':              'seq2seq_medium_cpu',
    'parameters':        '~245M',
    'device':            device,
    'prompt_format':     'IndicBARTSS ಪ್ರಶ್ನೆ+ಮಾಹಿತಿ </s> <2kn>',
    'reranking':         True,
    'eval_size':         len(eval_set),
    'avg_inference_sec': round(float(df['inference_sec'].mean()), 2),
    'bleu':              metrics['bleu'],
    'bertscore_f1':      metrics['bertscore_f1'],
    'answer_similarity': metrics['answer_similarity'],
    'context_relevance': metrics['context_relevance'],
    'faithfulness':      metrics['faithfulness'],
}

json_path = f'{RESULTS_DIR}/indicbartss_rag_metrics.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(result_dict, f, indent=2, ensure_ascii=False)
print(f'✅ Metrics JSON → {json_path}')
print()
print('─' * 58)
print('Summary:')
print(f'  Model     : {GEN_MODEL}')
print(f'  Size      : ~245M  (CPU-friendly)')
print(f'  BLEU      : {metrics["bleu"]}')
print(f'  BERTScore : {metrics["bertscore_f1"]}')
print('─' * 58)

✅ CSV saved → /content/drive/MyDrive/AgriQA/rag_comparison_results/indicbartss_rag_results.csv
✅ Metrics JSON → /content/drive/MyDrive/AgriQA/rag_comparison_results/indicbartss_rag_metrics.json

──────────────────────────────────────────────────────────
Summary:
  Model     : ai4bharat/IndicBARTSS
  Size      : ~245M  (CPU-friendly)
  BLEU      : 1.6402
  BERTScore : 0.6699
──────────────────────────────────────────────────────────
